In [ ]:
# -*- coding: utf-8 -*-

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# PARÂMETROS GERAIS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

SMOOTH_WIN = 5


# ============================================================
# PARÂMETROS DO RANDOM FOREST
# ============================================================

RF_COMP_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0
)


# ============================================================
# PARÂMETROS DO PARK
# ============================================================

PARK_MAX_SHIFT_FRAC = 0.25
PARK_NSTEPS = 201
PARK_SMOOTH_WIN = 5


# ============================================================
# PARÂMETROS DO AUTOENCODER
# ============================================================

AE_EPOCHS = 800
AE_BATCH_SIZE = 16
AE_LR = 1e-3
AE_LATENT_DIM = 64

AE_VAL_FRAC = 0.20
AE_PATIENCE = 120

ALPHA_COMP_AE = 0.85


# ============================================================
# PASTA DE SAÍDA
# ============================================================

OUTPUT_DIR = f"resultados_RMSD_CCDM_REF{REF_TEMP}C_{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ}kHz"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# SEMENTES
# ============================================================

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# -*- coding: utf-8 -*-

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# PARÂMETROS GERAIS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

SMOOTH_WIN = 5


# ============================================================
# PARÂMETROS DO RANDOM FOREST
# ============================================================

RF_COMP_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0
)


# ============================================================
# PARÂMETROS DO PARK
# ============================================================

PARK_MAX_SHIFT_FRAC = 0.25
PARK_NSTEPS = 201
PARK_SMOOTH_WIN = 5


# ============================================================
# PARÂMETROS DO AUTOENCODER
# ============================================================

AE_EPOCHS = 800
AE_BATCH_SIZE = 16
AE_LR = 1e-3
AE_LATENT_DIM = 64

AE_VAL_FRAC = 0.20
AE_PATIENCE = 120

ALPHA_COMP_AE = 0.85


# ============================================================
# PASTA DE SAÍDA
# ============================================================

OUTPUT_DIR = f"resultados_RMSD_CCDM_REF{REF_TEMP}C_{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ}kHz"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# SEMENTES
# ============================================================

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# ============================================================
# FUNÇÕES GERAIS
# ============================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []

    for c in df.columns:
        f = extract_freq_hz(c)

        if f is not None and fmin_khz <= f / 1e3 <= fmax_khz:
            cols.append(c)
            freqs.append(f)

    order = np.argsort(freqs)

    return [cols[i] for i in order], np.array(freqs, float)[order]


def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")

    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")

    return smooth[:len(arr)]


def add_extra_features_matrix(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)

    return np.hstack([X, mu, sd, amp])


def shift_interp(x, f, tau):
    f_shift = f + tau
    return np.interp(f, f_shift, x, left=x[0], right=x[-1])


def get_reference_curve(df, fcols, ref_temp):
    df_sem = df[df["falha"] == 0].copy()

    pool = df_sem.loc[
        np.isclose(df_sem["temperatura_c"], ref_temp),
        fcols
    ].to_numpy(float)

    if len(pool) == 0:
        raise ValueError(
            f"Nenhuma curva sem falha encontrada em REF_TEMP = {ref_temp} °C."
        )

    y_ref = np.median(pool, axis=0)

    return y_ref


# ============================================================
# MÉTRICAS
# ============================================================

def rmsd(y, ref):
    y = np.asarray(y, float)
    ref = np.asarray(ref, float)

    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, float)
    ref = np.asarray(ref, float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2))) + 1e-18

    corr = num / den

    return float(1 - corr)


def calcular_metricas(df, fcols, y_ref, metodo):
    X = df[fcols].to_numpy(float)

    df2 = df.copy()
    df2["RMSD"] = [rmsd(x, y_ref) for x in X]
    df2["CCDM"] = [ccdm(x, y_ref) for x in X]
    df2["Metodo"] = metodo

    return df2

In [ ]:
# ============================================================
# PARK
# ============================================================

def park_single(x, ref, fHz):
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = 1e99
    best_tau = 0.0
    best_dS = 0.0

    for tau in np.linspace(-tau_max, tau_max, PARK_NSTEPS):
        x_shift = shift_interp(x, fHz, tau)

        dS = np.mean(ref - x_shift)

        y_try = x_shift + dS

        err = np.sum((ref - y_try) ** 2)

        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y = shift_interp(x, fHz, best_tau) + best_dS
    y = moving_average(y, PARK_SMOOTH_WIN)

    return y


def compensar_park(df, fcols, fHz, y_ref):
    print("\n🔹 Aplicando Park...")

    X_all = df[fcols].to_numpy(float)

    Y_comp = np.zeros_like(X_all)

    for i in range(len(X_all)):
        Y_comp[i] = park_single(X_all[i], y_ref, fHz)

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    return df_comp

In [ ]:
# ============================================================
# RANDOM FOREST DIRETO
# ============================================================

def compensar_rf_direto(df, fcols, fHz, y_ref):
    print("\n🔹 Treinando Random Forest direto...")

    df_sem = df[df["falha"] == 0].copy()

    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    Y_target = y_ref[None, :] - X_sem

    X_aug_sem = add_extra_features_matrix(X_sem)
    X_in_sem = np.hstack([X_aug_sem, T_sem.reshape(-1, 1)])

    rf = RandomForestRegressor(**RF_COMP_PARAMS)
    rf.fit(X_in_sem, Y_target)

    print("🔹 Aplicando Random Forest direto...")

    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)

    X_aug_all = add_extra_features_matrix(X_all)
    X_in_all = np.hstack([X_aug_all, T_all.reshape(-1, 1)])

    Delta_hat = rf.predict(X_in_all)

    Y_comp = X_all + Delta_hat

    for i in range(len(Y_comp)):
        Y_comp[i] = moving_average(Y_comp[i], SMOOTH_WIN)

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    return df_comp, rf

In [ ]:
# ============================================================
# AUTOENCODER RESIDUAL
# ============================================================

class ResidualAutoencoder(nn.Module):
    def __init__(self, n_in, n_out, latent_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(n_in, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, n_out)
        )

    def forward(self, x):
        z = self.encoder(x)
        delta = self.decoder(z)

        return delta


def train_autoencoder_residual(
    X_input,
    Y_target,
    epochs=800,
    batch_size=16,
    lr=1e-3,
    latent_dim=64,
    val_frac=0.20,
    patience=120
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"\nDispositivo usado no Autoencoder: {device}")

    sx = StandardScaler()
    sy = StandardScaler()

    Xs = sx.fit_transform(X_input)
    Ys = sy.fit_transform(Y_target)

    N = Xs.shape[0]

    idx = np.arange(N)
    np.random.shuffle(idx)

    n_val = int(np.floor(val_frac * N))

    val_idx = idx[:n_val]
    train_idx = idx[n_val:]

    if len(val_idx) == 0:
        val_idx = train_idx.copy()

    X_train = torch.tensor(Xs[train_idx], dtype=torch.float32)
    Y_train = torch.tensor(Ys[train_idx], dtype=torch.float32)

    X_val = torch.tensor(Xs[val_idx], dtype=torch.float32)
    Y_val = torch.tensor(Ys[val_idx], dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train, Y_train),
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(X_val, Y_val),
        batch_size=batch_size,
        shuffle=False
    )

    model = ResidualAutoencoder(
        n_in=X_input.shape[1],
        n_out=Y_target.shape[1],
        latent_dim=latent_dim
    ).to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-5
    )

    loss_fn = nn.MSELoss()

    best_val = np.inf
    best_state = copy.deepcopy(model.state_dict())
    epochs_sem_melhora = 0

    for ep in range(1, epochs + 1):

        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()

            pred = model(xb)
            loss = loss_fn(pred, yb)

            loss.backward()
            opt.step()

            train_loss += loss.item()

        train_loss /= max(1, len(train_loader))

        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)

                val_loss += loss.item()

        val_loss /= max(1, len(val_loader))

        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_sem_melhora = 0
        else:
            epochs_sem_melhora += 1

        if ep % 50 == 0 or ep == 1:
            print(
                f"Epoch {ep:4d}/{epochs} | "
                f"train_loss = {train_loss:.6f} | "
                f"val_loss = {val_loss:.6f}"
            )

        if epochs_sem_melhora >= patience:
            print(
                f"Early stopping na epoch {ep}. "
                f"Melhor val_loss = {best_val:.6f}"
            )
            break

    model.load_state_dict(best_state)

    return model, sx, sy, device


def predict_autoencoder_residual(model, sx, sy, device, X_input):
    model.eval()

    Xs = sx.transform(X_input)
    X_tensor = torch.tensor(Xs, dtype=torch.float32).to(device)

    with torch.no_grad():
        pred_scaled = model(X_tensor).cpu().numpy()

    Delta_hat = sy.inverse_transform(pred_scaled)

    return Delta_hat


def compensar_autoencoder(df, fcols, fHz, y_ref):
    print("\n🔹 Treinando Autoencoder residual...")

    df_sem = df[df["falha"] == 0].copy()

    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    Y_target = y_ref[None, :] - X_sem

    X_aug_sem = add_extra_features_matrix(X_sem)
    X_in_sem = np.hstack([X_aug_sem, T_sem.reshape(-1, 1)])

    ae, sx, sy, device = train_autoencoder_residual(
        X_in_sem,
        Y_target,
        epochs=AE_EPOCHS,
        batch_size=AE_BATCH_SIZE,
        lr=AE_LR,
        latent_dim=AE_LATENT_DIM,
        val_frac=AE_VAL_FRAC,
        patience=AE_PATIENCE
    )

    print("🔹 Aplicando Autoencoder residual...")

    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)

    X_aug_all = add_extra_features_matrix(X_all)
    X_in_all = np.hstack([X_aug_all, T_all.reshape(-1, 1)])

    Delta_hat = predict_autoencoder_residual(
        ae,
        sx,
        sy,
        device,
        X_in_all
    )

    Y_comp = X_all + ALPHA_COMP_AE * Delta_hat

    for i in range(len(Y_comp)):
        Y_comp[i] = moving_average(Y_comp[i], SMOOTH_WIN)

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    return df_comp, ae

In [ ]:
# ============================================================
# EXECUÇÃO GERAL
# ============================================================

def executar_comparacao():
    timings = {}

    print("====================================================")
    print("COMPARAÇÃO PARK vs RF vs AUTOENCODER")
    print("====================================================")

    # -----------------------------
    # Carregar base
    # -----------------------------
    t0 = time.time()

    df = pd.read_pickle(ARQ_BASE)

    fcols, fhz = get_freq_columns(
        df,
        FREQ_MIN_KHZ,
        FREQ_MAX_KHZ
    )

    y_ref = get_reference_curve(
        df,
        fcols,
        REF_TEMP
    )

    timings["load_reference"] = time.time() - t0

    print(f"\nTotal de amostras: {len(df)}")
    print(f"Amostras sem falha: {len(df[df['falha'] == 0])}")
    print(f"Faixa usada: {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    print(f"Número de pontos de frequência: {len(fcols)}")
    print(f"Temperatura de referência: {REF_TEMP} °C")

    # -----------------------------
    # Original
    # -----------------------------
    t0 = time.time()

    df_original = calcular_metricas(
        df,
        fcols,
        y_ref,
        metodo="Original"
    )

    timings["original_metrics"] = time.time() - t0

    # -----------------------------
    # Park
    # -----------------------------
    t0 = time.time()

    df_park_raw = compensar_park(
        df,
        fcols,
        fhz,
        y_ref
    )

    df_park = calcular_metricas(
        df_park_raw,
        fcols,
        y_ref,
        metodo="Park"
    )

    timings["park"] = time.time() - t0

    # -----------------------------
    # Random Forest
    # -----------------------------
    t0 = time.time()

    df_rf_raw, rf_model = compensar_rf_direto(
        df,
        fcols,
        fhz,
        y_ref
    )

    df_rf = calcular_metricas(
        df_rf_raw,
        fcols,
        y_ref,
        metodo="RF Direto"
    )

    timings["rf_direto"] = time.time() - t0

    # -----------------------------
    # Autoencoder
    # -----------------------------
    t0 = time.time()

    df_ae_raw, ae_model = compensar_autoencoder(
        df,
        fcols,
        fhz,
        y_ref
    )

    df_ae = calcular_metricas(
        df_ae_raw,
        fcols,
        y_ref,
        metodo="Autoencoder"
    )

    timings["autoencoder"] = time.time() - t0

    # -----------------------------
    # Juntar tudo
    # -----------------------------
    df_long = pd.concat(
        [
            df_original,
            df_park,
            df_rf,
            df_ae
        ],
        axis=0,
        ignore_index=False
    )

    print("\n==================== TEMPOS ====================")
    for k, v in timings.items():
        print(f"{k:20s}: {v:.3f} s")

    print("\n✅ Comparação concluída.")

    return (
        df,
        df_original,
        df_park,
        df_rf,
        df_ae,
        df_long,
        y_ref,
        fcols,
        fhz,
        timings
    )

In [ ]:
# ============================================================
# RESUMOS NUMÉRICOS
# ============================================================

def resumo_geral(df_long, metodos=None):
    if metodos is not None:
        df_use = df_long[df_long["Metodo"].isin(metodos)].copy()
    else:
        df_use = df_long.copy()

    tabela = (
        df_use
        .groupby(["Metodo", "falha"])[["RMSD", "CCDM"]]
        .agg(["mean", "std", "min", "max"])
        .round(6)
    )

    return tabela


def resumo_por_temperatura(df_long, metodos=None):
    if metodos is not None:
        df_use = df_long[df_long["Metodo"].isin(metodos)].copy()
    else:
        df_use = df_long.copy()

    tabela = (
        df_use
        .groupby(["Metodo", "temperatura_c", "falha"])[["RMSD", "CCDM"]]
        .mean()
        .reset_index()
        .sort_values(["Metodo", "temperatura_c", "falha"])
    )

    return tabela


def checar_monotonicidade(df_long, metodos=None):
    if metodos is not None:
        df_use = df_long[df_long["Metodo"].isin(metodos)].copy()
    else:
        df_use = df_long.copy()

    registros = []

    for metodo in sorted(df_use["Metodo"].unique()):
        df_m = df_use[df_use["Metodo"] == metodo]

        temps = sorted(df_m["temperatura_c"].unique())

        for T in temps:
            df_t = df_m[np.isclose(df_m["temperatura_c"], T)]

            danos_disponiveis = set(df_t["falha"].unique())

            if not {0, 1, 2}.issubset(danos_disponiveis):
                continue

            for metrica in ["RMSD", "CCDM"]:
                medias = {}

                for d in [0, 1, 2]:
                    medias[d] = df_t.loc[df_t["falha"] == d, metrica].mean()

                ok = medias[0] < medias[1] < medias[2]

                registros.append({
                    "Metodo": metodo,
                    "Temperatura": T,
                    "Metrica": metrica,
                    "D0": medias[0],
                    "D1": medias[1],
                    "D2": medias[2],
                    "Monotonico_D0_D1_D2": ok
                })

    df_mono = pd.DataFrame(registros)

    if len(df_mono) == 0:
        print("Nenhuma temperatura possui D0, D1 e D2 simultaneamente.")
        return df_mono, None

    resumo = (
        df_mono
        .groupby(["Metodo", "Metrica"])["Monotonico_D0_D1_D2"]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={"Monotonico_D0_D1_D2": "Percentual_monotonico_%"} )
    )

    return df_mono, resumo

In [ ]:
# ============================================================
# ESTILO DOS GRÁFICOS
# ============================================================

def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 18,
        "axes.labelsize": 20,
        "axes.titlesize": 20,
        "xtick.labelsize": 17,
        "ytick.labelsize": 17,
        "legend.fontsize": 15,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


def plot_metricas_por_temperatura(
    df_long,
    dano=0,
    metodos=("Park", "RF Direto", "Autoencoder"),
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[
        (df_long["falha"] == dano) &
        (df_long["Metodo"].isin(metodos))
    ].copy()

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), dpi=300)

    metricas = ["RMSD", "CCDM"]

    for ax, metrica in zip(axes, metricas):

        for metodo in metodos:
            df_m = df_use[df_use["Metodo"] == metodo]

            g = (
                df_m
                .groupby("temperatura_c")[metrica]
                .mean()
                .reset_index()
                .sort_values("temperatura_c")
            )

            ax.plot(
                g["temperatura_c"],
                g[metrica],
                marker="o",
                linewidth=2,
                label=metodo
            )

        ax.set_xlabel("Temperatura (°C)")
        ax.set_ylabel(metrica)

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_title(f"(a) RMSD — Dano {dano}")
    axes[1].set_title(f"(b) CCDM — Dano {dano}")

    axes[0].legend(frameon=True, facecolor="white", edgecolor="none")

    plt.tight_layout()

    if salvar:
        nome = f"Metricas_por_temperatura_Dano{dano}.png"
        caminho = os.path.join(OUTPUT_DIR, nome)
        plt.savefig(caminho, bbox_inches="tight", facecolor="white")
        print(f"Figura salva em: {caminho}")

    plt.show()

In [ ]:
def plot_metricas_medias_por_dano(
    df_long,
    metodos=("Park", "RF Direto", "Autoencoder"),
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    danos = sorted(df_use["falha"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), dpi=300)

    metricas = ["RMSD", "CCDM"]

    x = np.arange(len(danos))
    bar_w = 0.25

    offsets = np.linspace(
        -bar_w,
        bar_w,
        len(metodos)
    )

    for ax, metrica in zip(axes, metricas):

        for i, metodo in enumerate(metodos):
            vals = []

            for d in danos:
                mask = (
                    (df_use["Metodo"] == metodo) &
                    (df_use["falha"] == d)
                )

                vals.append(df_use.loc[mask, metrica].mean())

            ax.bar(
                x + offsets[i],
                vals,
                width=bar_w,
                edgecolor="black",
                linewidth=0.6,
                label=metodo
            )

        ax.set_xlabel("Classe de dano")
        ax.set_ylabel(metrica)
        ax.set_xticks(x)
        ax.set_xticklabels([f"Dano {d}" for d in danos])

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_title("(a) RMSD médio por dano")
    axes[1].set_title("(b) CCDM médio por dano")

    axes[0].legend(frameon=True, facecolor="white", edgecolor="none")

    plt.tight_layout()

    if salvar:
        nome = "Metricas_medias_por_dano.png"
        caminho = os.path.join(OUTPUT_DIR, nome)
        plt.savefig(caminho, bbox_inches="tight", facecolor="white")
        print(f"Figura salva em: {caminho}")

    plt.show()

In [ ]:
def plot_curva_exemplo(
    df_base,
    df_park,
    df_rf,
    df_ae,
    y_ref,
    fcols,
    fhz,
    idx_show=None,
    falha=None,
    temperatura=None,
    salvar=True
):
    aplicar_estilo_artigo()

    if idx_show is None:
        df_aux = df_base.copy()

        if falha is not None:
            df_aux = df_aux[df_aux["falha"] == falha]

        if temperatura is not None:
            df_aux = df_aux[np.isclose(df_aux["temperatura_c"], temperatura)]

        if len(df_aux) == 0:
            raise ValueError("Nenhuma curva encontrada com os filtros dados.")

        idx_show = df_aux.index[0]

    fhz_khz = fhz / 1e3

    T = df_base.loc[idx_show, "temperatura_c"]
    D = df_base.loc[idx_show, "falha"]

    plt.figure(figsize=(12, 6), dpi=300)

    plt.plot(
        fhz_khz,
        y_ref,
        "--",
        linewidth=1.2,
        label=f"Referência {REF_TEMP} °C"
    )

    plt.plot(
        fhz_khz,
        df_base.loc[idx_show, fcols],
        linewidth=1.0,
        alpha=0.70,
        label=f"Original — {T} °C — Dano {D}"
    )

    plt.plot(
        fhz_khz,
        df_park.loc[idx_show, fcols],
        linewidth=1.5,
        label="Park"
    )

    plt.plot(
        fhz_khz,
        df_rf.loc[idx_show, fcols],
        linewidth=1.5,
        label="RF Direto"
    )

    plt.plot(
        fhz_khz,
        df_ae.loc[idx_show, fcols],
        linewidth=1.5,
        label="Autoencoder"
    )

    plt.xlabel("Frequência (kHz)")
    plt.ylabel("Parte real da impedância")

    plt.title(
        f"Comparação de compensação — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz"
    )

    plt.legend(frameon=True, facecolor="white", edgecolor="none")
    plt.grid(False)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    if salvar:
        nome = f"Curva_exemplo_idx{idx_show}_Dano{D}_Temp{T}.png"
        caminho = os.path.join(OUTPUT_DIR, nome)
        plt.savefig(caminho, bbox_inches="tight", facecolor="white")
        print(f"Figura salva em: {caminho}")

    plt.show()

    print(f"Índice usado: {idx_show}")
    print(f"Temperatura: {T} °C")
    print(f"Dano: {D}")

In [ ]:
(
    df_base,
    df_original,
    df_park,
    df_rf,
    df_ae,
    df_long,
    y_ref,
    fcols,
    fhz,
    timings
) = executar_comparacao()

In [ ]:
metodos_comp = ["Park", "RF Direto", "Autoencoder"]

print("\n================ RESUMO GERAL ================")
tabela_resumo = resumo_geral(df_long, metodos=metodos_comp)
display(tabela_resumo)


print("\n================ RESUMO POR TEMPERATURA ================")
tabela_temp = resumo_por_temperatura(df_long, metodos=metodos_comp)
display(tabela_temp)


print("\n================ MONOTONICIDADE D0 < D1 < D2 ================")
df_mono, resumo_mono = checar_monotonicidade(df_long, metodos=metodos_comp)

display(df_mono)
display(resumo_mono)

In [ ]:
# Métricas por temperatura para dano 0
plot_metricas_por_temperatura(
    df_long,
    dano=0,
    metodos=("Park", "RF Direto", "Autoencoder"),
    salvar=True
)

# Métricas por temperatura para dano 1
plot_metricas_por_temperatura(
    df_long,
    dano=1,
    metodos=("Park", "RF Direto", "Autoencoder"),
    salvar=True
)

# Métricas por temperatura para dano 2
plot_metricas_por_temperatura(
    df_long,
    dano=2,
    metodos=("Park", "RF Direto", "Autoencoder"),
    salvar=True
)

# Médias por dano
plot_metricas_medias_por_dano(
    df_long,
    metodos=("Park", "RF Direto", "Autoencoder"),
    salvar=True
)

In [ ]:
plot_curva_exemplo(
    df_base=df_base,
    df_park=df_park,
    df_rf=df_rf,
    df_ae=df_ae,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=0,
    temperatura=48,
    salvar=True
)

In [ ]:
plot_curva_exemplo(
    df_base=df_base,
    df_park=df_park,
    df_rf=df_rf,
    df_ae=df_ae,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=1,
    temperatura=55,
    salvar=True
)

In [ ]:
plot_curva_exemplo(
    df_base=df_base,
    df_park=df_park,
    df_rf=df_rf,
    df_ae=df_ae,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=2,
    temperatura=55,
    salvar=True
)

In [ ]:
resumo_mono